In [0]:
import pyspark
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,DoubleType,DateType,TimestampType

In [0]:
catalog_name = 'oag'

In [0]:
def read_bronze():
    bronze_df = spark.table(f"{catalog_name}.bronze.bronze_table")
    return bronze_df

In [0]:
def drop_columns(df):
    silver_df = df.drop("ingestion_timestamp", "source_system", "operation_type")
    return silver_df

In [0]:
def drop_duplicates(df):
    silver_df = df.dropDuplicates(["transaction_id"])
    return silver_df

In [0]:
def validate_transaction_date(df):
    df = df.withColumn(
        "transaction_date",
        F.to_date(F.col("transaction_date"), "yyyy-MM-dd")
    )
    valid_df = df.filter(
        F.col("transaction_date").isNotNull() &
        (F.col("transaction_date") <= F.current_date())
    )
    return valid_df     

In [0]:
def validate_scores(df):
    valid_score_df = df.filter(F.col('supplier_reliability_score').between(0,1))
    valid_df = valid_score_df.filter(F.col("quality_score").between(0,100))
    return valid_df

In [0]:
def handle_category_missing_values(df):
    category_columns = ["product_name","product_category","quantity_unit","refinery_name","destination_city","transportation_mode"]
    for column in category_columns:
        df = df.withColumn(column,\
            F.when(F.col(column).isNull()|(F.trim(F.col(column)) == ""),F.lit("Unknown"))\
                .otherwise(F.col(column))
        )
    return df

In [0]:
def standard_columns(df):
    columns = ["product_name","product_category","quantity_unit","refinery_name","quality_status","disruption_type","delivery_status"]
    for i in columns:
        df = df.withColumn(i,F.initcap(F.trim(F.col(i))))
    return df

In [0]:
def create_silver_table(df):
    df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.silver_table")
    return True

In [0]:
df = read_bronze()
silver_df_rows = df.count()
silver_df_columns = len(df.columns)
print("Rows and columns",(silver_df_rows,silver_df_columns))

In [0]:
silver_new_df = drop_columns(df)
silver_df_columns = len(silver_new_df.columns)
print("No of columns:",silver_df_columns)

In [0]:
silver_df = drop_duplicates(silver_new_df)
silver_df_rows = silver_df.count()
silver_df_columns = len(silver_df.columns)
print("Rows and columns",(silver_df_rows,silver_df_columns))

In [0]:
silver_valid_date_df = validate_transaction_date(silver_df)
silver_df_rows = silver_valid_date_df.count()
print("No of records:",silver_df_rows)

In [0]:
silver_valid_score_df = validate_scores(silver_valid_date_df)
ilver_df_rows = silver_valid_date_df.count()
print("No of records:",silver_df_rows)
silver_df.limit(5).display()


In [0]:
silver_df = handle_category_missing_values(silver_df)
silver_df.limit(5).display()

In [0]:
silver_df = standard_columns(silver_valid_score_df)
silver_df.limit(5).display()

In [0]:
if create_silver_table(silver_df):
    print("silver table created")
else:
    print("Silver table creation failed")